# Treino da CNN — Contador e Classificador de GrãosTrabalho prático de Processamento de Imagens e Visão Computacional (PIVC 2026/2).Este notebook treina a rede que separa **grão inteiro** de **grão quebrado/defeituoso**.Ele roda no Google Colab e usa o mesmo código de `src/train.py`, para que o modelotreinado aqui funcione sem ajuste na aplicação local.## Antes de começarNa sua máquina, gere os recortes a partir das fotos dos lotes:```python -m src.dataset data/fotos --saida data/recortes```Confira as pastas `data/recortes/inteiro` e `data/recortes/defeituoso` e mova osrecortes que o baseline classificou errado. Depois compacte a pasta:```data/recortes  ->  recortes.zip```É esse `recortes.zip` que você envia no passo 2.

## 1. Clonar o repositório do projeto

In [ ]:
!git clone https://github.com/llstrehl/CCC_Processamento_De_Imagens.git
%cd CCC_Processamento_De_Imagens
!pip install -q opencv-python

## 2. Enviar o `recortes.zip` com o dataset já conferido

In [ ]:
from google.colab import files
import zipfile, pathlib

enviados = files.upload()           # selecione o recortes.zip
nome = next(iter(enviados))

destino = pathlib.Path('data/recortes')
destino.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(nome) as z:
    z.extractall('data')

for classe in ['inteiro', 'defeituoso']:
    pasta = destino / classe
    print(classe, '->', len(list(pasta.glob('*.png'))), 'recortes')

## 3. Conferir o equilíbrio entre as classesSe uma classe tiver muito mais exemplos que a outra, a rede tende a chutar sempre amajoritária. O ideal é ter pelo menos umas 100 imagens de cada e uma proporção quenão passe de três para um.

In [ ]:
import matplotlib.pyplot as plt
import cv2, pathlib

fig, eixos = plt.subplots(2, 6, figsize=(12, 4))
for linha, classe in enumerate(['inteiro', 'defeituoso']):
    arquivos = sorted(pathlib.Path(f'data/recortes/{classe}').glob('*.png'))[:6]
    for coluna, arquivo in enumerate(arquivos):
        imagem = cv2.cvtColor(cv2.imread(str(arquivo)), cv2.COLOR_BGR2RGB)
        eixos[linha][coluna].imshow(imagem)
        eixos[linha][coluna].axis('off')
    eixos[linha][0].set_title(classe, loc='left')
plt.tight_layout()
plt.show()

## 4. Treinar

In [ ]:
from src.train import train

historico = train(pasta_dados='data/recortes', saida='models/grain_cnn.keras')

## 5. Ver as curvas de treinoSe a curva de validação descolar muito da de treino, há decoreba: colete mais fotos.

In [ ]:
import matplotlib.pyplot as plt

fig, (esq, dir) = plt.subplots(1, 2, figsize=(11, 4))
esq.plot(historico.history['accuracy'], label='treino')
esq.plot(historico.history['val_accuracy'], label='validação')
esq.set_title('Acurácia'); esq.set_xlabel('época'); esq.legend()

dir.plot(historico.history['loss'], label='treino')
dir.plot(historico.history['val_loss'], label='validação')
dir.set_title('Perda'); dir.set_xlabel('época'); dir.legend()
plt.tight_layout(); plt.show()

## 6. Baixar o modeloSalve o arquivo em `models/grain_cnn.keras` no projeto local. A aplicação passa a usar a CNN automaticamente na próxima vez que abrir.

In [ ]:
from google.colab import files
files.download('models/grain_cnn.keras')